In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import os
from row_generate_func import *
from auto_insert_func import *
from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [3]:
# --- Main workflow ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
file_name = 'Fern_raw_db_station_matched.csv'
df_match = pd.read_csv(output_folder + file_name)

df_match

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaN,NaN,NaN,NaN,batch2
2,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaN,NaN,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaN,NaN,NaN,NaN,batch2
4,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaN,NaN,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
529,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
530,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
531,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1


In [4]:
station_names = df_match['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id, m.lat, m.lon, m.elev
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    db_stations = pd.read_sql(query, conn, params={"station_names": station_names})

In [5]:
merged = df_match.merge(
    db_stations,
    on='station_name',
    how='left'
)

match_info = merged.rename(
    columns={
        'native_id': 'native_id_db',
        'lat_y': 'lat_db',
        'lon_y': 'lon_db',
        'elev_y': 'elev_db'
    }
)

match_info = match_info.drop(columns=['native_id_raw', 'lat_x', 'lon_x', 'elev_x'])



In [6]:
match_info[['station_name', 'variable']]

,station_name,variable
0,Atlin School,DewPt
1,Atlin School,Gust Speed
2,Atlin School,RH
3,Atlin School,Rain
4,Atlin School,Sfc Temp
...,...,...
528,Willow-BowronWx,Solar Radiation
529,Willow-BowronWx,Temp
530,Willow-BowronWx,Wetness
531,Willow-BowronWx,Wind Direction


In [7]:
import re

# 1️⃣ Define base variable
def base_variable(var):
    return re.sub(r'\s*\d+$', '', var).strip()

match_info['base_variable'] = match_info['variable'].apply(base_variable)

# 2️⃣ Filter rows with duplicates per station + base_variable + unit_raw
dup_items = (
    match_info
    .groupby(['station_name', 'base_variable', 'unit_raw'])
    .filter(lambda x: len(x) > 1)
    .reset_index(drop=True)
)

dup_items

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db,base_variable
0,Atlin School,Atlin School,Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,TempC,celsius,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0,Temp
1,Atlin School,Atlin School,Temp 2,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,NaN,NaN,NaN,NaN,NaN,NaN,batch5,AtlSch,59.574000,-133.687000,705.0,Temp
2,BlackhawkWx,BlackhawkWx,DewPt,°C,2012-05-24 13:00:00,2026-01-05 10:00:00,°C,DewPtC,celsius,2012-05-24 13:00:00,2024-01-05 08:00:00,0.0,731.0,batch1,Blackhawk,55.078885,-120.352171,962.0,DewPt
3,BlackhawkWx,BlackhawkWx,DewPt 2,°C,2024-07-18 10:00:00,2025-07-24 09:00:00,°C,NaN,NaN,NaN,NaN,NaN,NaN,batch5,Blackhawk,55.078885,-120.352171,962.0,DewPt
4,BlackhawkWx,BlackhawkWx,RH,%,2012-05-24 13:00:00,2026-01-05 10:00:00,%,RH,%,2012-05-24 13:00:00,2024-01-05 08:00:00,0.0,731.0,batch1,Blackhawk,55.078885,-120.352171,962.0,RH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,ThompsonWx,ThompsonWx,DewPt 2,°C,2024-09-24 12:00:00,2025-09-22 10:00:00,°C,NaN,NaN,NaN,NaN,NaN,NaN,batch5,Thompson,54.333115,-126.099445,869.0,DewPt
110,ThompsonWx,ThompsonWx,RH,%,2007-09-12 17:00:00,2025-09-22 10:00:00,%,RH,%,2007-09-12 16:00:00,2023-09-13 11:00:00,0.0,739.0,batch1,Thompson,54.333115,-126.099445,869.0,RH
111,ThompsonWx,ThompsonWx,RH 2,%,2024-09-24 12:00:00,2025-09-22 10:00:00,%,NaN,NaN,NaN,NaN,NaN,NaN,batch5,Thompson,54.333115,-126.099445,869.0,RH
112,ThompsonWx,ThompsonWx,Temp,°C,2007-09-12 17:00:00,2025-09-22 10:00:00,°C,TempC,celsius,2007-09-12 16:00:00,2023-09-13 11:00:00,0.0,739.0,batch1,Thompson,54.333115,-126.099445,869.0,Temp


2 sensors for the same station variable, the first sensors value has been inserted. Now for more records have only the second sensor or both, I insert them or update them. For the item with only the second sensor, seemes insert successufully. So need to update the values only from one sensor to the mean of two sensors. So I need to update the values of that obs_raw values. Right now I just need to update the obs_raw.

In [14]:
all_rows = []
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'
network_name = "FLNRO-FERN"

stations = dup_items.groupby(['station_name', 'base_variable'])

for (station_name, base_var), group in stations:
    
    print(f"Processing station: {station_name}, variable: {base_var}")

    csv_file_name = group['station_file_name'].iloc[0] + '.csv'
    csv_path = os.path.join(data_path, csv_file_name)
    df_data = pd.read_csv(csv_path, encoding='latin1', low_memory=False)

    time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    unit_raw = group['unit_raw'].iloc[0]
    variable_cols = [f"{v}, {unit_raw}" for v in group['variable'] if f"{v}, {unit_raw}" in df_data.columns]

    if not variable_cols or len(variable_cols) < 2:
        # Skip if only one sensor exists
        print(f"  ⚠ Only one sensor for {base_var}, skipping.")
        continue

    # Compute row-wise mean across sensors
    df_data[variable_cols] = df_data[variable_cols].apply(
        pd.to_numeric,
        errors='coerce'   # non-numeric → NaN
    )
    
    df_data['value'] = df_data[variable_cols].mean(axis=1, skipna=True)

    # Create a column showing which sensors contributed
    def contributing_sensors(row):
        sensors = [col for col in variable_cols if pd.notna(row[col])]
        if len(sensors) > 1:
            return f"mean of {', '.join(sensors)}"
        elif sensors:
            return sensors[0]
        else:
            return None

    df_data['sensor_source'] = df_data.apply(contributing_sensors, axis=1)

    # --- Filter: skip rows that only come from the first sensor ---
    # Assume the first sensor column is variable_cols[0]
    df_data = df_data[~(df_data['sensor_source'] == variable_cols[0])]
    df_data = df_data.dropna(subset=['value']).reset_index(drop=True)

    # --- Check for differences between sensors ---
    # Only consider rows where both sensor values exist
    sensor_diff_rows = df_data.dropna(subset=variable_cols)
    
    # Count how many rows have different values across sensors
    # Use `nunique` row-wise
    sensor_diff_count = (sensor_diff_rows[variable_cols].nunique(axis=1) > 1).sum()
    
    print(f"  🔹 Rows with differing sensor values: {sensor_diff_count} out of {len(sensor_diff_rows)}")

    print(f"{len(df_data)} values from {df_data['sensor_source'].unique()}")

    if df_data.empty:
        print(f"  ⚠ No rows left after skipping first sensor for {base_var}")
        continue

    # --- Build Row objects ---
    for _, row_i in df_data.iterrows():
        all_rows.append(Row(
            time=row_i[time_col],
            val=row_i['value'],
            variable_name=group['db_var_match'].iloc[0],
            unit=group['unit_db'].iloc[0],
            network_name=network_name,
            station_id=group['native_id_db'].iloc[0],
            lat=float(group['lat_db'].iloc[0]),
            lon=float(group['lon_db'].iloc[0])
        ))

all_rows = [r for r in all_rows if not pd.isna(r.time)]

Processing station: Atlin School, variable: Temp
  ⚠ Only one sensor for Temp, skipping.
Processing station: BlackhawkWx, variable: DewPt
  ⚠ Only one sensor for DewPt, skipping.
Processing station: BlackhawkWx, variable: RH
  🔹 Rows with differing sensor values: 8900 out of 8903
8904 values from ['mean of RH, %, RH 2, %' 'RH 2, %']
Processing station: BlackhawkWx, variable: Temp
  ⚠ Only one sensor for Temp, skipping.
Processing station: BowronPit, variable: DewPt
  ⚠ Only one sensor for DewPt, skipping.
Processing station: BowronPit, variable: RH
  🔹 Rows with differing sensor values: 7770 out of 7774
7774 values from ['mean of RH, %, RH 2, %']
Processing station: BowronPit, variable: Temp
  ⚠ Only one sensor for Temp, skipping.
Processing station: Canoe Mountain Stn, variable: DewPt
  ⚠ Only one sensor for DewPt, skipping.
Processing station: Canoe Mountain Stn, variable: RH
  🔹 Rows with differing sensor values: 17482 out of 17495
17495 values from ['mean of RH, %, RH 2, %']
Proces

In [13]:
all_rows

[Row(time=Timestamp('2024-07-18 10:00:00'), val=5.8385, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, lon=-120.352171),
 Row(time=Timestamp('2024-07-18 11:00:00'), val=6.317, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, lon=-120.352171),
 Row(time=Timestamp('2024-07-18 12:00:00'), val=6.413, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, lon=-120.352171),
 Row(time=Timestamp('2024-07-18 13:00:00'), val=7.232, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, lon=-120.352171),
 Row(time=Timestamp('2024-07-18 14:00:00'), val=6.4125, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, lon=-120.352171),
 Row(time=Timestamp('2024-07-18 15:00:00'), val=7.4925, variable_name='RH', unit='%', network_name='FLNRO-FERN', station_id='Blackhawk', lat=55.078885, l

In [10]:
len(all_rows)

268163

In [11]:
import math

all_valid = all(
    (r.val is not None) and (not math.isnan(r.val))
    for r in all_rows
)

print(all_valid)

True


In [15]:
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"

safe_insert_rows(
    all_rows,
    log_file_path=output_folder + 'fern_all_station_insert.log',
    db_url=db_url,
    chunk_size=5000,        # SAFE starting point
    bulk_chunk_size=1000,   # internal DB bulk insert size
)

🚀 Starting insert: 268163 rows in 54 chunks
➡️  Processing rows 0–4999
{"asctime": "2026-02-24 22:52:43,057", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2026-02-24 22:52:43,574", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:52:44,021", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:52:44,502", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:52:44,907", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:52:45,353", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 5000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026

#### batch update

In [16]:
update_sql = sa.text("""
UPDATE obs_raw AS o
SET datum = :val
FROM meta_history AS m,
     meta_station AS s,
     meta_vars AS v
WHERE o.history_id = m.history_id
  AND m.station_id = s.station_id
  AND o.vars_id = v.vars_id
  AND s.native_id = :station_id
  AND v.net_var_name = :variable_name
  AND o.obs_time = :obs_time
  AND o.datum IS DISTINCT FROM :val
""")

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield i, lst[i:i + size]

BATCH_SIZE = 3000
total = len(dup_rows)

for start, chunk in chunked(dup_rows, BATCH_SIZE):
    params = [
        {
            "val": float(r.val),
            "station_id": r.station_id,
            "variable_name": r.variable_name,
            "obs_time": r.time,
        }
        for r in chunk
    ]

    with engine.begin() as conn:
        result = conn.execute(update_sql, params)

    end = min(start + BATCH_SIZE, total)
    print(
        f"Updated batch {start}-{end - 1} "
        f"({result.rowcount} rows updated)"
    )

Updated batch 0-2999 (2976 rows updated)
Updated batch 3000-5999 (2970 rows updated)
Updated batch 6000-8999 (2980 rows updated)
Updated batch 9000-11999 (2973 rows updated)
Updated batch 12000-14999 (2981 rows updated)
Updated batch 15000-17999 (2985 rows updated)
Updated batch 18000-20999 (2986 rows updated)
Updated batch 21000-23999 (2967 rows updated)
Updated batch 24000-26999 (2975 rows updated)
Updated batch 27000-29999 (2966 rows updated)
Updated batch 30000-32999 (2979 rows updated)
Updated batch 33000-35999 (2991 rows updated)
Updated batch 36000-38999 (2989 rows updated)
Updated batch 39000-41999 (2973 rows updated)
Updated batch 42000-44999 (2975 rows updated)
Updated batch 45000-47999 (2963 rows updated)
Updated batch 48000-50999 (2971 rows updated)
Updated batch 51000-53999 (2970 rows updated)
Updated batch 54000-56999 (2966 rows updated)
Updated batch 57000-59999 (2971 rows updated)
Updated batch 60000-62999 (2977 rows updated)
Updated batch 63000-65999 (2975 rows updated